# 🔬 Notebook 3: Reddit — Deep Dives


## 🎯 Learning objectives

- Understand the **Hot ranking** formula piece by piece and see why it beats naïve alternatives.
- See a real lock-contention bottleneck on a single counter and fix it with **sharded counters**.
- Serve a comment tree efficiently: compare recursive parent_id queries to **materialized path**
  and **closure tables**.
- Every deep-dive follows the repo convention: **bad → better → best** with runnable code.


## 🛠️ Setup

```bash
cd 06-system-designs/reddit
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1️⃣ Hot ranking — what goes to the top?

A Hot feed must balance **quality** (lots of upvotes) and **freshness** (posted recently).
If we pick either signal alone we lose.


### 🐌 Bad — sort by net upvotes


In [ ]:
import random, math, time

random.seed(7)

# 10 posts: (id, ups, downs, age_hours)
posts = [
    ("A-brand-new",       5,    0,  0.0),   # new, few votes
    ("B-one-hour",       80,    5,  1.0),
    ("C-six-hours",     500,   40,  6.0),
    ("D-one-day",      3000,  200, 24.0),
    ("E-three-days",  20000, 1000, 72.0),   # old but huge
    ("F-controversial", 300,  290,  2.0),
    ("G-fresh-good",    150,    2,  0.5),
    ("H-week-old",    50000,  500, 24*7.),  # a week old classic
    ("I-meh",             8,    6, 10.0),
    ("J-rising",         60,    1,  0.25),
]

print("SORT BY NET UPVOTES ONLY (ignores age):")
for pid, u, d, age_h in sorted(posts, key=lambda p: -(p[1]-p[2])):
    print(f"  net={u-d:>6}  age={age_h:>5.1f}h  {pid}")


Problem: **H-week-old** and **E-three-days** dominate forever. A brand-new viral post
(J-rising) looks tiny next to them. Reddit would ossify.


### ✅ Better — Hacker News style exponential decay


In [ ]:
# Hacker News: score = (ups - 1) / (age_hours + 2) ** 1.8
def hn_score(ups, downs, age_h):
    return (max(ups - downs - 1, 0)) / ((age_h + 2) ** 1.8)

print("HACKER-NEWS SCORE (exponential decay):")
for pid, u, d, age_h in sorted(posts, key=lambda p: -hn_score(p[1], p[2], p[3])):
    print(f"  score={hn_score(u,d,age_h):>8.3f}  age={age_h:>5.1f}h  {pid}")


Much better — recent posts can dethrone old classics. But decay is *continuous*: every
minute that passes recomputes every post's score, otherwise the ranking drifts. Expensive.


### 🏆 Best — Reddit's log + post-time formula


In [ ]:
# The actual Reddit hot formula uses the post's creation time measured from a
# fixed epoch (e.g. Reddit's launch, 2005-12-08) — NOT the post's age-from-now:
#
#   t  = seconds_between(post_created_at, EPOCH)      # newer post → larger t
#   score = log10(max(|ups - downs|, 1))  +  sign(ups - downs) * t / 45_000
#
# Three key properties this gives us:
#   • log-scaling: 10,000 votes is not 10x better than 1,000 — it's +1 on the score.
#   • Newer posts start with a HIGHER baseline (larger t), so fresh content can rise.
#   • The score of an EXISTING post never changes for fixed votes — so when a new vote
#     arrives we only re-score THAT post and nudge it in the sorted list. No global
#     decay sweep. This is the trick that makes the Hot feed cheap to maintain.
#   • Net-downvoted posts get a NEGATIVE time bonus → they sink fast (anti-spam).

import time as _time

# Pretend "now" is the reference point; older posts have SMALLER t.
NOW_T = int(_time.time())           # seconds since unix epoch ("now")
def t_of(age_seconds: float) -> int:
    # post created (age_seconds) ago → t = NOW - age
    return NOW_T - int(age_seconds)

def hot_score(ups: int, downs: int, age_seconds: float) -> float:
    net = ups - downs
    sign = 1 if net > 0 else (-1 if net < 0 else 0)
    order = math.log10(max(abs(net), 1))
    t = t_of(age_seconds)            # seconds-since-epoch at post-creation time
    return round(order + sign * t / 45_000, 4)

print("REDDIT HOT SCORE (log(votes) + post-time/45000):")
for pid, u, d, age_h in sorted(posts, key=lambda p: -hot_score(p[1], p[2], p[3]*3600)):
    print(f"  score={hot_score(u,d,age_h*3600):>14.3f}  age={age_h:>5.1f}h  net={u-d:>6}  {pid}")


Observe the ordering:

- Fresh, well-upvoted posts (e.g. `J-rising`, `G-fresh-good`) now land near the top.
- Week-old classics (`H-week-old`) fall away, even though they have the most votes — their
  smaller `t` is a **12.5-hour-per-log10-votes** drag.
- Scores of *existing* posts don't change over time for fixed votes — so when a new vote
  lands, we only re-score *that* post. No global recompute. That's the whole trick.

### Sanity check on the *differences*

The raw scores look huge because they include the giant `seconds-since-epoch` term, but the
interesting thing is the **difference** between posts, which is what a sort uses. A one-log10
bump in votes buys you ≈ `45000 / 3600 ≈ 12.5 hours` of recency. Let's verify:


In [ ]:
# Two posts: A has 10x more votes than B but was posted 12.5h earlier.
# They should rank roughly equal.
a_score = hot_score(ups=10_000, downs=0, age_seconds=12.5*3600)   # older, many votes
b_score = hot_score(ups=1_000,  downs=0, age_seconds=0)           # brand-new, fewer votes
print(f"A (12.5h old, 10k votes): {a_score:.3f}")
print(f"B ( 0.0h old,  1k votes): {b_score:.3f}")
print(f"difference              : {b_score - a_score:+.3f}    (≈ 0 → roughly tied)")


## 2️⃣ Vote counter contention — the hot-key problem

When a post goes viral, millions of users hit the same row:

```
UPDATE posts SET ups = ups + 1 WHERE id = 42;   -- serialised on one row
```

Only one transaction at a time can hold the row lock. Every other vote queues up. Let's
measure it with threads + a single mutex as a stand-in for the row lock.


### 🐌 Bad — one global counter, one lock


In [ ]:
import threading, time

class SingleCounter:
    def __init__(self):
        self.n = 0
        self.lock = threading.Lock()
    def inc(self):
        with self.lock:                       # every thread fights for THIS lock
            # simulate the DB doing ~30 µs of work inside the critical section
            _ = sum(range(50))
            self.n += 1

def run(counter, n_threads, per_thread):
    t0 = time.time()
    threads = [threading.Thread(target=lambda: [counter.inc() for _ in range(per_thread)])
               for _ in range(n_threads)]
    for t in threads: t.start()
    for t in threads: t.join()
    return time.time() - t0

single = SingleCounter()
dt = run(single, n_threads=16, per_thread=20_000)
print(f"single counter : {single.n:>7,} votes in {dt:5.2f}s   → {single.n/dt:,.0f} votes/s")


### ✅ Better — sharded counters (one lock per shard)


In [ ]:
class ShardedCounter:
    def __init__(self, n_shards: int = 64):
        self.shards = [0] * n_shards
        self.locks  = [threading.Lock() for _ in range(n_shards)]

    def inc(self, user_id: int):
        idx = user_id % len(self.shards)      # pick shard by user id → even spread
        with self.locks[idx]:
            _ = sum(range(50))                # same simulated DB work
            self.shards[idx] += 1

    def total(self):
        return sum(self.shards)

sharded = ShardedCounter(n_shards=64)

def run_sharded(counter, n_threads, per_thread):
    t0 = time.time()
    def worker(tid):
        for i in range(per_thread):
            counter.inc(tid * 1000 + i)       # unique uid → spreads shards
    threads = [threading.Thread(target=worker, args=(t,)) for t in range(n_threads)]
    for t in threads: t.start()
    for t in threads: t.join()
    return time.time() - t0

dt = run_sharded(sharded, n_threads=16, per_thread=20_000)
print(f"64-shard counter: {sharded.total():>7,} votes in {dt:5.2f}s   → {sharded.total()/dt:,.0f} votes/s")


Sharding turns a single contended row into N less-contended rows. The trade-off is the
**read side**: to display `ups`, we must `SELECT SUM(ups) FROM vote_shards WHERE post_id=X`
— 64 reads instead of 1. But reads are **cached**, so this is a good trade.

### 🏆 Best — write-behind to Redis, batch-flush to DB

At Reddit scale even a sharded DB counter is too slow. The production pattern is:

```
vote → INCR hot:post:42:ups   (Redis, in-memory, microseconds)
         │
         ▼
   every N seconds  ─►  a flusher reads Redis counters
                         and does ONE batched UPDATE per post
```

- Writes are **non-blocking**: `INCR` in Redis is O(1) and uncontended.
- Reads hit Redis directly; the DB is only the source of truth for durability.
- If Redis loses a few seconds of votes, no one cries — vote counts are an estimate anyway.

Let's simulate the Redis pattern with a dict + lock-free-ish counter.


In [ ]:
# Stand-in for Redis INCR: atomic increment on an integer.
# threading.Lock is still used because Python ints aren't atomic, but the window is tiny.
class WriteBehindCounter:
    def __init__(self):
        self.buffer: dict[int, int] = {}
        self.lock = threading.Lock()
        self.durable: dict[int, int] = {}  # "the database"

    def inc(self, post_id: int):
        with self.lock:                    # trivial CAS-style section
            self.buffer[post_id] = self.buffer.get(post_id, 0) + 1

    def flush(self):
        with self.lock:
            delta = self.buffer
            self.buffer = {}
        # one bulk update — imagine this is a single SQL: UPDATE ... CASE WHEN id=?
        for pid, d in delta.items():
            self.durable[pid] = self.durable.get(pid, 0) + d

wb = WriteBehindCounter()
def run_wb(counter, n_threads, per_thread, post_id=42):
    t0 = time.time()
    threads = [threading.Thread(target=lambda: [counter.inc(post_id) for _ in range(per_thread)])
               for _ in range(n_threads)]
    for t in threads: t.start()
    for t in threads: t.join()
    return time.time() - t0

dt = run_wb(wb, n_threads=16, per_thread=20_000)
wb.flush()
print(f"write-behind   : {wb.durable[42]:>7,} votes in {dt:5.2f}s   → {wb.durable[42]/dt:,.0f} votes/s")
print("After flush, durable store shows:", wb.durable)


| Approach | Throughput | Consistency | Complexity |
|---|---|---|---|
| Single counter | 💀 low | strong | trivial |
| Sharded counter (64) | 🔺 high | strong | small |
| Redis write-behind | 🔥 highest | eventual (seconds) | flusher + failover |

Reddit/Twitter/YouTube-scale systems all use the write-behind approach for like/view counts.


## 3️⃣ Serving comment trees

Our `comments` table has both `parent_id` (adjacency list) and `path` (materialized path)
from Notebook 2. Let's see the three ways to load a subtree.


In [ ]:
# Fresh SQLite DB with a bigger fake tree so timing is meaningful.
import sqlite3, random, time

db = sqlite3.connect(":memory:")
db.row_factory = sqlite3.Row
db.executescript('''
    CREATE TABLE comments (
        id INTEGER PRIMARY KEY,
        post_id INTEGER,
        parent_id INTEGER,
        body TEXT,
        path TEXT
    );
    CREATE INDEX ix_comments_post_parent ON comments(post_id, parent_id);
    CREATE INDEX ix_comments_path        ON comments(path);

    -- Closure table: (ancestor, descendant, depth)
    CREATE TABLE comment_ancestry (
        ancestor   INTEGER,
        descendant INTEGER,
        depth      INTEGER,
        PRIMARY KEY (ancestor, descendant)
    );
''')

random.seed(1)

def insert_comment(post_id, parent_id, body):
    cur = db.execute("INSERT INTO comments(post_id, parent_id, body) VALUES (?,?,?)",
                     (post_id, parent_id, body))
    cid = cur.lastrowid
    if parent_id is None:
        path = f"/{cid}/"
    else:
        parent_path = db.execute("SELECT path FROM comments WHERE id=?", (parent_id,)).fetchone()["path"]
        path = f"{parent_path}{cid}/"
    db.execute("UPDATE comments SET path=? WHERE id=?", (path, cid))

    # closure-table rows: self + all ancestor rows
    db.execute("INSERT INTO comment_ancestry VALUES (?,?,0)", (cid, cid))
    if parent_id is not None:
        db.execute('''INSERT INTO comment_ancestry(ancestor, descendant, depth)
                      SELECT ancestor, ?, depth+1 FROM comment_ancestry WHERE descendant=?''',
                   (cid, parent_id))
    return cid

# Grow a random tree of 5 000 comments under post #1
roots = [insert_comment(1, None, f"root {i}") for i in range(20)]
all_ids = list(roots)
for i in range(5_000):
    parent = random.choice(all_ids)
    all_ids.append(insert_comment(1, parent, f"reply {i}"))
db.commit()
print("comments inserted:", db.execute("SELECT COUNT(*) FROM comments").fetchone()[0])


### 🐌 Bad — walk `parent_id` in Python (N+1 queries)


In [ ]:
def fetch_subtree_py(root_id: int):
    out = []
    stack = [root_id]
    while stack:
        cid = stack.pop()
        row = db.execute("SELECT id, body FROM comments WHERE id=?", (cid,)).fetchone()
        out.append(row["body"])
        kids = db.execute("SELECT id FROM comments WHERE parent_id=?", (cid,)).fetchall()
        stack.extend(k["id"] for k in kids)
    return out

root = roots[0]
t0 = time.time(); rows = fetch_subtree_py(root); dt = (time.time()-t0)*1000
print(f"N+1 walk         : {len(rows):>5} rows in {dt:6.1f} ms")


### ✅ Better — one indexed prefix query via materialized path


In [ ]:
def fetch_subtree_path(root_id: int):
    path = db.execute("SELECT path FROM comments WHERE id=?", (root_id,)).fetchone()["path"]
    return db.execute("SELECT body FROM comments WHERE path LIKE ? ORDER BY path",
                      (path + "%",)).fetchall()

t0 = time.time(); rows = fetch_subtree_path(root); dt = (time.time()-t0)*1000
print(f"materialized path: {len(rows):>5} rows in {dt:6.1f} ms")


### 🏆 Best — closure table join (flexible: filter by depth too)


In [ ]:
def fetch_subtree_closure(root_id: int, max_depth: int | None = None):
    q = '''SELECT c.body
           FROM comment_ancestry a
           JOIN comments c ON c.id = a.descendant
           WHERE a.ancestor = ?'''
    args = [root_id]
    if max_depth is not None:
        q += " AND a.depth <= ?"; args.append(max_depth)
    return db.execute(q, args).fetchall()

t0 = time.time(); rows = fetch_subtree_closure(root); dt = (time.time()-t0)*1000
print(f"closure table    : {len(rows):>5} rows in {dt:6.1f} ms")

# Closure wins when you want "only the first 3 levels of replies" —
# a materialized path can't do that without string-counting '/'.
t0 = time.time(); rows = fetch_subtree_closure(root, max_depth=2); dt = (time.time()-t0)*1000
print(f"closure (depth≤2): {len(rows):>5} rows in {dt:6.1f} ms")


### Pick-your-tree cheat sheet

| Pattern | Read subtree | Read level-limited | Move/rewrite a branch | Storage |
|---|---|---|---|---|
| `parent_id` only | ❌ recursion | ❌ recursion | ✅ cheap | 🟢 tiny |
| Materialized path | ✅ fast | ⚠️ string math | ❌ rewrite all descendants | 🟡 |
| Closure table | ✅ fast | ✅ trivial | ⚠️ rewrite ancestry rows | 🔴 largest |

In practice Reddit **caches** the rendered comment JSON per `(post_id, sort)` in Redis and
invalidates on new comment / vote. The DB layout above is for the *cache-miss* path.


## 📚 Summary

1. **Hot ranking = log(votes) + age_bonus.** The score of an existing post never changes for
   fixed votes, so we only re-score *new* votes. No expensive global decay pass.
2. **Don't fight a single row** under vote storms. Shard counters — or, even better, buffer
   in Redis and flush to the DB in batches. Eventual consistency is fine for vote counts.
3. **Don't walk parent pointers** on hot paths. Use a materialized path (fast subtree) or a
   closure table (flexible depth filters) — both turn a recursive problem into one index scan.

### 💡 Interview tips

- If asked "how do you rank Hot?", lead with: *"log(|ups − downs|) + sign × age / 45000 — the
  key property is that existing posts' scores don't change, so we only re-rank on new votes."*
- For vote contention, mention **sharded counters** *and* **write-behind** — showing both says
  you understand the levels of scale.
- For comments, mention **cached pre-rendered JSON per (post, sort)** — Reddit's secret sauce.

### Where to go next

- `04-patterns/` — Cache-Aside, Write-Behind, and Materialized View are the patterns powering this lab.
- `06-system-designs/fb-news-feed/` — different ranking problem (per-user relevance vs global popularity).
- `06-system-designs/top-k/` — generalises "Hot" to any streaming top-K problem.
